In [5]:
!apt-get update -qq && apt-get install -y ffmpeg -qq
!pip install -q yt-dlp
!pip install -q openai-whisper
!pip install -q gensim
!pip install -q tensorflow
!pip install -q pandas numpy openpyxl
!pip install -q gradio

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 50.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 12.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 60.0 MB/s eta 0:00:00


In [54]:
import os
import re
import whisper
import yt_dlp
import numpy as np
import gradio as gr
from tensorflow.keras.models import load_model
from gensim.models import FastText

# Load models
model_path = "/content/drive/MyDrive/Fitra_RL_Models/CNN_LSTM_reinforced.h5"
fasttext_path = "/content/drive/MyDrive/Fitra_RL_Models/fasttext_model.model"

print("⏳ Loading Fitra models...")
try:
    model = load_model(model_path)
    fasttext_model = FastText.load(fasttext_path)
    print("✅ Models loaded successfully.")
except Exception as e:
    print(f"⚠️ Error loading models: {e}")


def prepare_input(title_tokens=None, desc_tokens=None, transcript_tokens=None):
    def get_vector(tokens):
        vectors = [fasttext_model.wv[word] for word in tokens if word in fasttext_model.wv]
        return np.mean(vectors, axis=0) if vectors else np.zeros(300)

    title_vector = get_vector(title_tokens) if title_tokens else np.zeros(300)
    desc_vector = get_vector(desc_tokens) if desc_tokens else np.zeros(300)
    transcript_vector = get_vector(transcript_tokens) if transcript_tokens else np.zeros(300)

    combined = np.hstack([title_vector, desc_vector, transcript_vector])
    return combined.reshape(1, 900, 1)


def detect_language_with_whisper(audio_path):
    temp_model = whisper.load_model("tiny")
    result = temp_model.transcribe(audio_path, fp16=False, language=None)
    return result.get("language", "en")


def extract_transcript(video_url):
    audio_path = "audio.mp3"
    ydl_opts = {'format': 'bestaudio', 'outtmpl': audio_path}
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([video_url])
    except Exception as e:
        print(f"❌ Audio download error: {e}")
        return None

    detected_lang = detect_language_with_whisper(audio_path)
    whisper_model_size = "medium" if detected_lang == "ar" else "base"
    dynamic_model = whisper.load_model(whisper_model_size)
    result = dynamic_model.transcribe(audio_path, fp16=False, language=detected_lang)

    if os.path.exists(audio_path):
        os.remove(audio_path)
    return result["text"]


def is_arabic(text):
    if not isinstance(text, str):
        return False
    return any('\u0600' <= char <= '\u06FF' for char in text)


def get_video_metadata(video_url):
    ydl_opts = {'quiet': True, 'skip_download': True}
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(video_url, download=False)
            return info.get('title', ''), info.get('description', '')
    except:
        return "", ""


def tokenize(text):
    return re.sub(r"[^\w\s]", "", text).lower().split() if isinstance(text, str) else []


def analyze_video(video_url):
    if not video_url:
        return "الرجاء إدخال رابط أولاً!"

    transcript = extract_transcript(video_url)
    if transcript is None:
        return "⚠️ فشل التحليل (الفيديو غير متاح)"

    if is_arabic(transcript):
        title_tokens, desc_tokens = [], []
    else:
        title, description = get_video_metadata(video_url)
        title_tokens = tokenize(title)
        desc_tokens = tokenize(description)

    transcript_tokens = tokenize(transcript)
    x_input = prepare_input(title_tokens, desc_tokens, transcript_tokens)

    prediction = model.predict(x_input)[0][0]
    return "شاذ" if prediction > 0.5 else "غير شاذ"


custom_css = """
body, .gradio-container {
    background-color: #FBF9FC !important;
}

.container {
    background-color: #FBF9FC;
    padding: 25px;
    border-radius: 16px;
}

.title-area {
    font-family: 'Segoe UI', Arial, sans-serif;
    color: #4A5568;
    font-size: 26px;
    font-weight: 600;
    margin-bottom: 4px;
}

.subtitle-area {
    font-family: 'Segoe UI', Arial, sans-serif;
    color: #718096;
    font-size: 14px;
    margin-bottom: 20px;
}

.style-textbox > div {
    background-color: #FFFFFF !important;
    border: 0.5px solid #313A46 !important;
    border-radius: 100px !important;
    padding: 2px !important;
    margin-bottom: 15px;
}

.style-textbox label span {
    color: #4A5568 !important;
    font-weight: 600 !important;
    font-size: 15px !important;
    margin-bottom: 6px !important;
    display: inline-block !important;
}

.style-textbox > .wrap,
.style-textbox .wrap,
.style-textbox > div,
.gradio-textbox > .wrap {
    border: none !important;
    box-shadow: none !important;
    background: transparent !important;
    padding: 0 !important;
}

.style-textbox textarea,
.style-textbox input[type="text"],
.style-textbox input {
    background-color: #FFFFFF !important;
    color: #2D3748 !important;
    font-size: 16px !important;
    border: 0.5px solid #E2E8F0 !important;
    border-radius: 10px !important;
    padding: 14px !important;
    width: 100% !important;
    box-shadow: none !important;
    outline: none !important;
    -webkit-appearance: none !important;
    -moz-appearance: none !important;
    appearance: none !important;
    resize: none !important;
}

.style-textbox textarea:disabled,
.style-textbox input:disabled {
    color: #2D3748 !important;
    -webkit-text-fill-color: #2D3748 !important;
}

.style-textbox textarea::placeholder,
.style-textbox input::placeholder {
    color: #A0AEC0 !important;
}

.style-textbox textarea:focus,
.style-textbox input:focus {
    border: 0.5px solid #CBD5E0 !important;
    outline: none !important;
    box-shadow: none !important;
}

input[type=number]::-webkit-outer-spin-button,
input[type=number]::-webkit-inner-spin-button {
    -webkit-appearance: none !important;
    margin: 0 !important;
    display: none !important;
}
input[type=number] {
    -moz-appearance: textfield !important;
}

textarea::-webkit-scrollbar { width: 4px !important; }
textarea::-webkit-scrollbar-thumb { background: #E2E8F0 !important; border-radius: 4px !important; }
textarea::-webkit-scrollbar-track { background: transparent !important; }

button.primary-btn {
    background-color: #FFFFFF !important;
    color: #4A5568 !important;
    border: 1px solid #E2E8F0 !important;
    border-radius: 24px !important;
    font-weight: 600 !important;
    width: 160px !important;
    margin: 20px auto !important;
    display: block !important;
    box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.05) !important;
}

button.primary-btn:hover {
    background-color: #F7FAFC !important;
    border-color: #CBD5E0 !important;
}
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Default()) as demo:
    with gr.Column(elem_classes="container"):
        gr.HTML("<div class='title-area'>⚙ Analyze YouTube Video</div>")
        gr.HTML("<div class='subtitle-area'>Add a URL to analyze content:</div>")

        url_input = gr.Textbox(
            label="",
            placeholder="Enter URL",
            show_label=False,
            lines=1,
            elem_classes="style-textbox"
        )

        analyze_btn = gr.Button("Analyze", elem_classes="primary-btn")

        output_text = gr.Textbox(
            label="Result / النتيجة",
            show_label=True,
            interactive=False,
            elem_classes="style-textbox"
        )

        analyze_btn.click(fn=analyze_video, inputs=url_input, outputs=output_text)

demo.launch(inline=True, share=False ,  height=700)

⏳ Loading Fitra models...
✅ Models loaded successfully.


/tmp/ipykernel_550/466073452.py:224: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Default()) as demo:
/tmp/ipykernel_550/466073452.py:224: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Default()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>